# 02 · 数据清洗

本 Notebook 将原始营销数据处理为可复用的建模数据集，并保留完整清洗审计记录。

## 清洗决策

- 移除完全重复记录。
- 原数据没有空值，因此不进行缺失值填补。
- 将 `unknown` 保留为显式类别：它代表客户信息未知，不应错误地改为众数。
- 保留原始 `y`，并新增数值化目标变量 `target_subscription`（`yes=1`，`no=0`）。
- `duration` 暂时保留在清洗数据中；仅在通话前预测模型训练时排除。

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import clean_bank_marketing_data, load_bank_marketing_data, save_dataframe

RAW_DATA_PATH = PROJECT_ROOT / 'data/raw/bank-additional/bank-additional/bank-additional-full.csv'
CLEAN_DATA_PATH = PROJECT_ROOT / 'data/processed/bank_marketing_clean.csv'
AUDIT_PATH = PROJECT_ROOT / 'data/processed/cleaning_audit.csv'

In [2]:
raw_df = load_bank_marketing_data(RAW_DATA_PATH)
clean_df, cleaning_audit = clean_bank_marketing_data(raw_df)

display(cleaning_audit)
print(f'原始数据：{raw_df.shape[0]:,} 行 × {raw_df.shape[1]} 列')
print(f'清洗数据：{clean_df.shape[0]:,} 行 × {clean_df.shape[1]} 列')

,metric,value
0,rows_before_cleaning,41188
1,duplicate_rows_removed,12
2,rows_after_cleaning,41176
3,missing_cells_after_cleaning,0
4,unknown_category_cells_retained,12716


原始数据：41,188 行 × 21 列
清洗数据：41,176 行 × 22 列


## `unknown` 类别审计

以下统计用于后续决定是否需要将“未知”信息单独作为特征。

In [3]:
string_columns = clean_df.select_dtypes(include=['object', 'string']).columns
unknown_summary = pd.DataFrame({
    'unknown_count': (clean_df[string_columns] == 'unknown').sum(),
    'unknown_rate': (clean_df[string_columns] == 'unknown').mean(),
}).query('unknown_count > 0').sort_values('unknown_count', ascending=False)
display(unknown_summary.style.format({'unknown_rate': '{:.2%}'}))

,unknown_count,unknown_rate
default,8596,20.88%
education,1730,4.20%
housing,990,2.40%
loan,990,2.40%
job,330,0.80%
marital,80,0.19%


## 保存清洗产物

In [4]:
save_dataframe(clean_df, CLEAN_DATA_PATH)
save_dataframe(cleaning_audit, AUDIT_PATH)

reloaded_df = pd.read_csv(CLEAN_DATA_PATH)
assert reloaded_df.shape == clean_df.shape
assert reloaded_df['target_subscription'].isin([0, 1]).all()
print(f'已保存清洗数据：{CLEAN_DATA_PATH.relative_to(PROJECT_ROOT)}')
print(f'已保存清洗审计：{AUDIT_PATH.relative_to(PROJECT_ROOT)}')

已保存清洗数据：data\processed\bank_marketing_clean.csv
已保存清洗审计：data\processed\cleaning_audit.csv


## 下一步

进入 `03_eda.ipynb`，分析客户属性、联系活动和宏观经济指标与订购行为的关系。